# Model Serving Platform: FastAPI + Docker + Monitoring

[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/genieincodebottle/model-serving-platform)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/genieincodebottle/aiml-companion/blob/main/projects/model-serving-platform/notebooks/Model_Serving_Platform.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-View_Source-blue?logo=github)](https://github.com/genieincodebottle/aiml-companion/tree/main/projects/model-serving-platform)

**Author:** [genieincodebottle](https://github.com/genieincodebottle) | **Last Updated:** March 2026

> Build production ML infrastructure: model server, health checks, metrics, and load testing.

<div class="alert alert-info">
<strong>What you will learn:</strong>
<ul>
<li>Train and serialize a model for production serving</li>
<li>Build <strong>FastAPI-style</strong> predict and health endpoints</li>
<li>Implement <strong>latency metrics</strong> collection (P50/P95/P99)</li>
<li>Run <strong>concurrent load testing</strong> to benchmark throughput</li>
</ul>
</div>

<div class="alert alert-warning">
<strong>Note:</strong> Full Docker/CI deployment runs locally. This notebook demos the core components (model server, metrics, load testing) that run anywhere.
</div>

---

<a id="toc"></a>
## Table of Contents

1. [Setup](#setup) - Install scikit-learn, joblib
2. [Train & Save Model](#train) - Iris classifier with Random Forest
3. [Model Server](#server) - Predict and health check endpoints
4. [Metrics Collection](#metrics) - Latency tracking (P50/P95/P99)
5. [Load Test Simulation](#load-test) - 500 concurrent requests
6. [Docker & CI/CD](#docker) - Full deployment guide
7. [Key Takeaways](#takeaways) - Production patterns

---

<a id="setup"></a>
## 1. Setup

In [ ]:
!pip install scikit-learn joblib numpy -q

---

<a id="train"></a>
## 2. Train & Save Model

Train a Random Forest classifier on the Iris dataset and save it with joblib for production loading.

In [ ]:
import numpy as np, joblib, os, time
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)
print(f"Accuracy: {model.score(X_test, y_test):.4f}")

os.makedirs("models", exist_ok=True)
joblib.dump(model, "models/model.joblib")
print("Model saved")

<div class="alert alert-success">
<strong>Model trained and saved.</strong> The Iris dataset is a simple 3-class problem (150 samples, 4 features). Random Forest achieves near-perfect accuracy on this dataset. The model is serialized with joblib for fast loading at server startup.
</div>

---

<a id="server"></a>
## 3. Model Server (FastAPI Components)

<div class="alert alert-info">
<strong>Production pattern:</strong> Load the model <strong>once at startup</strong> (not per-request). The <code>predict()</code> function returns predictions, probabilities, and model version - matching the structure of a real FastAPI endpoint.
</div>

In [ ]:
model_server = joblib.load("models/model.joblib")
model_version = "v1.0.0"
start_time = time.time()

def predict(features):
    arr = np.array(features).reshape(1, -1)
    pred = int(model_server.predict(arr)[0])
    prob = float(model_server.predict_proba(arr).max())
    return {"prediction": pred, "probability": prob, "model_version": model_version}

def health():
    return {"status": "healthy", "model_loaded": True, "version": model_version,
            "uptime_s": round(time.time() - start_time, 1)}

print("Health:", health())
print("Predict:", predict([5.1, 3.5, 1.4, 0.2]))

<div class="alert alert-success">
<strong>Server ready.</strong> The predict endpoint returns class prediction, probability, and model version. Health endpoint confirms the model is loaded and reports uptime. In production, these would be FastAPI routes behind a load balancer.
</div>

---

<a id="metrics"></a>
## 4. Metrics Collection

<div class="alert alert-info">
<strong>Why percentiles, not averages?</strong> Average latency hides tail latency spikes. P99 (99th percentile) shows the worst-case experience for 1 in 100 users - this is what matters for SLA compliance.
</div>

In [ ]:
from collections import defaultdict

class Metrics:
    def __init__(self):
        self.latencies = []
        self.predictions = defaultdict(int)
        self.errors = defaultdict(int)
    def record(self, latency, pred_class, version):
        self.latencies.append(latency)
        self.predictions[f"{version}:{pred_class}"] += 1
    def report(self):
        lats = sorted(self.latencies)
        return {"total": len(lats),
                "p50_ms": round(np.percentile(lats, 50)*1000, 2),
                "p95_ms": round(np.percentile(lats, 95)*1000, 2),
                "p99_ms": round(np.percentile(lats, 99)*1000, 2)}

metrics = Metrics()
for _ in range(100):
    features = [np.random.normal(loc, 0.5) for loc in [5.8, 3.0, 3.7, 1.2]]
    t0 = time.time()
    r = predict(features)
    metrics.record(time.time() - t0, r["prediction"], r["model_version"])

print("Metrics:", metrics.report())

<div class="alert alert-success">
<strong>Metrics collected from 100 warmup requests:</strong>
<ul>
<li>Latency percentiles (P50, P95, P99) give a complete picture of response time distribution</li>
<li>Prediction class distribution tracks if the model is producing balanced outputs</li>
<li>In production, export these to Prometheus/Grafana for real-time dashboards</li>
</ul>
</div>

---

<a id="load-test"></a>
## 5. Load Test Simulation

Simulate production traffic: 500 requests across 10 threads with a 90/10 predict/health split (realistic traffic pattern).

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import random

def sim_request():
    if random.random() < 0.1:
        t0 = time.time()
        health()
        return ("health", time.time() - t0)
    features = [random.gauss(0, 1) for _ in range(4)]
    t0 = time.time()
    predict(features)
    return ("predict", time.time() - t0)

print("Running 500 requests across 10 threads...")
with ThreadPoolExecutor(max_workers=10) as ex:
    results = list(ex.map(lambda _: sim_request(), range(500)))

pred_lats = sorted([lat for typ, lat in results if typ == "predict"])
print(f"Predictions: {len(pred_lats)}")
print(f"P50: {np.percentile(pred_lats, 50)*1000:.1f}ms")
print(f"P95: {np.percentile(pred_lats, 95)*1000:.1f}ms")
print(f"P99: {np.percentile(pred_lats, 99)*1000:.1f}ms")
print(f"Throughput: {len(pred_lats)/sum(pred_lats):.0f} req/s")

<div class="alert alert-success">
<strong>Load Test Results:</strong>
<ul>
<li><strong>P50</strong> (median) shows typical request latency - should be sub-millisecond for a simple model</li>
<li><strong>P95</strong> shows that 95% of requests complete within this time</li>
<li><strong>P99</strong> shows worst-case tail latency - spikes here indicate GC pauses or resource contention</li>
<li><strong>Throughput</strong> (req/s) shows how many concurrent requests the server can handle</li>
<li>In production, these numbers would include network latency, serialization, and model inference time</li>
</ul>
</div>

---

<a id="docker"></a>
## 6. Docker & CI/CD (Run Locally)

```bash
git clone https://github.com/genieincodebottle/aiml-companion.git
cd aiml-companion/projects/model-serving-platform
docker build -t ml-server -f docker/Dockerfile .
docker run -p 8000:8000 ml-server
# Then: curl localhost:8000/health
```

<div class="alert alert-info">
The repo includes Dockerfile, docker-compose.yml, CI/CD workflow, and Locust load tests for full production deployment.
</div>

---

<a id="takeaways"></a>
## 7. Key Takeaways

<div class="alert alert-success">
<strong>Production patterns demonstrated:</strong>
<ul>
<li><strong>Lifespan loading</strong> - Load model once at startup, not per-request (eliminates cold-start latency)</li>
<li><strong>Health checks</strong> - Docker HEALTHCHECK + /health endpoint for orchestrator liveness probes</li>
<li><strong>Graceful shutdown</strong> - SIGTERM handler for completing in-flight requests before stopping</li>
<li><strong>P50/P95/P99 metrics</strong> - Track tail latency, not just averages, for SLA compliance</li>
<li><strong>Load testing</strong> - 90/10 predict/health split simulates realistic production traffic</li>
</ul>
</div>

<div class="alert alert-info">
<strong>Next steps:</strong>
<ul>
<li>Add <strong>model versioning</strong> with A/B testing (canary deployments)</li>
<li>Implement <strong>Prometheus metrics</strong> for real-time monitoring dashboards</li>
<li>Add <strong>request validation</strong> with Pydantic models</li>
<li>Set up <strong>Locust load testing</strong> for sustained throughput benchmarks</li>
</ul>
</div>

---

<div class="alert alert-info">
<strong>Found this notebook useful?</strong> Give it an upvote on Kaggle! Have questions or suggestions? Leave a comment below or open an issue on <a href="https://github.com/genieincodebottle/aiml-companion">GitHub</a>.
</div>